# Prediction-Level Audit and Table Verification

This notebook creates the prediction-level audit artifacts requested for
independent verification. It reads the released test predictions, split
assignments, training weights, and LICE-pattern trace. It does not retrain any
model and does not rerun LIME or DiCE.

The automated verification reloads the released prediction file and
reconstructs confusion matrices, full-precision metrics, false-negative
transitions, McNemar results, and the stable workflow outputs.

## 1. Connect Google Drive and install the environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

configured_root = os.environ.get("LICE_PROJECT_ROOT")
project_candidates = [Path(configured_root) if configured_root else None, Path.cwd(), Path("/content/drive/MyDrive/Research/LICE_Guided_Model_Refinement_v1.3.0"), Path("/content/drive/MyDrive/LICE_Guided_Model_Refinement_v1.3.0")]
PROJECT_ROOT = next((p.resolve() for p in project_candidates if p and (p / "data").exists()), None)
assert PROJECT_ROOT is not None, "Repository not found; set LICE_PROJECT_ROOT."

from importlib.metadata import PackageNotFoundError, version
import sys

PINNED_BINARY_STACK = {
    "numpy": ("numpy", "2.5.1"),
    "pandas": ("pandas", "2.2.3"),
    "scikit-learn": ("sklearn", "1.5.2"),
    "scipy": ("scipy", "1.18.0"),
    "statsmodels": ("statsmodels", "0.14.4"),
}


def installed_version(distribution_name):
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return None


restart_required = any(
    installed_version(distribution) != required
    or (
        module in sys.modules
        and getattr(sys.modules[module], "__version__", None) != required
    )
    for distribution, (module, required) in PINNED_BINARY_STACK.items()
)

%pip install -q -r "{PROJECT_ROOT / 'environment' / 'requirements.txt'}"

if restart_required:
    print(
        "Pinned packages were installed. Colab will restart now. "
        "After it reconnects, run this setup cell once more and "
        "then continue to the next cell.",
        flush=True,
    )
    os.kill(os.getpid(), 9)

print(f"Release folder: {PROJECT_ROOT}")

## 2. Build the audit artifacts and run automated verification

The verification program is intentionally independent of the in-memory model
training workflow. A failure raises an exception and prevents a successful
completion message.

In [ ]:
import runpy

verification_program = (
    PROJECT_ROOT / "tests" / "verify_original_results_stable_names.py"
)

assert verification_program.exists(), verification_program

verification_namespace = runpy.run_path(
    str(verification_program),
    run_name="__main__",
)

del verification_namespace

## 3. Inspect the verification outputs

In [ ]:
import json

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

results_dir = PROJECT_ROOT / "results" / "test_ablation"
checks = pd.read_csv(results_dir / "prediction_reproduction_checks.csv")
transitions = pd.read_csv(
    results_dir / "fn_transition_summary_from_predictions.csv"
)
with (results_dir / "prediction_audit_manifest.json").open(
    encoding="utf-8"
) as stream:
    manifest = json.load(stream)

assert checks["Status"].eq("PASS").all()
assert manifest["all_checks_passed"] is True
assert manifest["row_counts"]["test_instances"] == 13812
assert manifest["row_counts"]["model_variants"] == 8
assert manifest["row_counts"]["long_prediction_rows"] == 110496
assert manifest["row_counts"]["training_audit_rows"] == 55245

display(transitions)
display(checks[["Check_ID", "Scope", "Status"]])

print(
    f"Checks passed: {checks['Status'].eq('PASS').sum()} / {len(checks)}"
)
print("All prediction-level artifacts and the stable workflow outputs were verified.")

## Interpretation of fold and sample-weight fields

The 13,812 evaluation observations belong to the untouched outer test set and
therefore do not have OOF validation-fold assignments. Their evaluation weight
is 1.0 for every model. The model-specific Mild, Balanced, and High-Sensitivity
weights apply only to training observations and are provided with OOF fold and
LICE-pattern fields in `predictions/oof/training_lice_assignment_audit.csv.gz`.